In [5]:
#Cody's Info
#Steam API Key: D055A982E240438C0651DD3B290894B3 - ISteamuserstats
#Steam ID: 76561199121994793

#Matt's Info
#Steam API Key: 8718028FBB50D15D16CE98F13624C382 - ISteamuser
#Steam ID: 76561198244049512

from google.colab import drive
drive.mount('/content/drive')

# Set your output folder
output_dir = "/content/drive/MyDrive/Data Wrangling Project/"

import os
os.makedirs(output_dir, exist_ok=True)
import os
print(f"Working directory: {os.getcwd()}")

import requests
import pandas as pd
import time

# Configuration — Matt & Cody only
users = [
    {"steam_id": "76561199121994793", "api_key": "D055A982E240438C0651DD3B290894B3", "label": "Cody"},
    {"steam_id": "76561198244049512", "api_key": "8718028FBB50D15D16CE98F13624C382", "label": "Matt"}
]

# DATA COLLECTION FUNCTIONS

def get_player_summaries(steam_ids, api_key):
    chunks = [steam_ids[i:i+100] for i in range(0, len(steam_ids), 100)]
    all_players = []
    for chunk in chunks:
        url = "http://api.steampowered.com/ISteamUser/GetPlayerSummaries/v2/"
        params = {"key": api_key, "steamids": ",".join(chunk)}
        response = requests.get(url, params=params).json()
        players = response["response"].get("players", [])
        all_players.extend(players)
    df = pd.DataFrame(all_players)
    for col in ["steamid", "personaname", "loccountrycode", "timecreated"]:
        if col not in df.columns:
            df[col] = None
    df = df[["steamid", "personaname", "loccountrycode", "timecreated"]].rename(columns={
        "personaname": "username",
        "loccountrycode": "country"
    })
    return df

def get_owned_games(steam_ids, api_key):
    all_dfs = []
    url = "http://api.steampowered.com/IPlayerService/GetOwnedGames/v1/"
    for sid in steam_ids:
        params = {"key": api_key, "steamid": sid, "include_appinfo": True, "include_played_free_games": True}
        response = requests.get(url, params=params).json()
        games = response["response"].get("games", [])
        if not games:
            print(f"No data for SteamID {sid} — profile may be private or empty.")
            continue
        temp = pd.DataFrame(games)
        temp["steamid"] = sid
        all_dfs.append(temp)
        time.sleep(0.5)
    if not all_dfs:
        return pd.DataFrame()
    df = pd.concat(all_dfs, ignore_index=True)
    df["playtime_hours"] = df["playtime_forever"] / 60
    df = df[df["playtime_hours"] > 0]
    df = df.drop(columns=["playtime_forever"])
    df = df[["steamid", "appid", "name", "playtime_hours"]]
    return df

def get_achievements(steam_ids, app_ids, api_key):
    all_dfs = []
    for sid in steam_ids:
        for appid in app_ids:
            url = "http://api.steampowered.com/ISteamUserStats/GetPlayerAchievements/v1/"
            params = {"key": api_key, "steamid": sid, "appid": appid}
            response = requests.get(url, params=params).json()
            stats = response.get("playerstats", {})
            if not stats.get("success") or "achievements" not in stats:
                continue
            achievements = pd.DataFrame(stats["achievements"])
            completed = achievements[achievements["achieved"] == 1].shape[0]
            total = achievements.shape[0]
            all_dfs.append({"steamid": sid, "appid": appid,
                           "achievements_completed": completed, "achievements_total": total})
            time.sleep(0.5)
    if not all_dfs:
        return pd.DataFrame(columns=["steamid", "appid", "achievements_completed", "achievements_total"])
    return pd.DataFrame(all_dfs)

def get_steamspy_data(app_ids):
    all_dfs = []
    for appid in app_ids:
        url = f"https://steamspy.com/api.php?request=appdetails&appid={appid}"
        response = requests.get(url).json()
        try:
            price_usd = int(response.get("price", 0)) / 100
        except (ValueError, TypeError):
            price_usd = None
        all_dfs.append({
            "appid": appid, "steamspy_name": response.get("name"),
            "genre": response.get("genre"), "tags": response.get("tags"),
            "owners": response.get("owners"), "positive_ratings": response.get("positive"),
            "negative_ratings": response.get("negative"), "price_usd": price_usd,
            "discount": response.get("discount"), "average_playtime": response.get("average_forever")
        })
        time.sleep(1)
    return pd.DataFrame(all_dfs)

# BUILD RAW MASTER DATAFRAME

def build_master_df(users):
    all_frames = []
    for user in users:
        sid, key, label = user["steam_id"], user["api_key"], user["label"]
        print(f"\nPulling data for {label}...")
        players_df = get_player_summaries([sid], key)
        games_df = get_owned_games([sid], key)
        if games_df.empty:
            print(f"  No game data for {label}, skipping.")
            continue
        app_ids = games_df["appid"].unique().tolist()
        achievements_df = get_achievements([sid], app_ids, key)
        steamspy_df = get_steamspy_data(app_ids)
        df = games_df.merge(players_df, on="steamid", how="left")
        df = df.merge(achievements_df, on=["steamid", "appid"], how="left")
        df = df.merge(steamspy_df, on="appid", how="left")
        df["label"] = label
        all_frames.append(df)
    all_frames = [f for f in all_frames if not f.empty]
    return pd.concat(all_frames, ignore_index=True)

master_df = build_master_df(users)

print(f"\n{'='*60}")
print(f"RAW DATA — NO WRANGLING YET")
print(f"{'='*60}")
print(f"Shape: {master_df.shape}")
print(f"Columns: {list(master_df.columns)}")
print(f"\nDtypes:\n{master_df.dtypes}")
print(f"\nMissing values:\n{master_df.isnull().sum()}")
print(f"\nFirst 3 rows:")
print(master_df.head(3).to_string())

master_df.to_csv(f"{output_dir}steam_v1_raw.csv", index=False)
print("\nExported: steam_v1_raw.csv")

Mounted at /content/drive
Working directory: /content

Pulling data for Cody...

Pulling data for Matt...

RAW DATA — NO WRANGLING YET
Shape: (130, 19)
Columns: ['steamid', 'appid', 'name', 'playtime_hours', 'username', 'country', 'timecreated', 'achievements_completed', 'achievements_total', 'steamspy_name', 'genre', 'tags', 'owners', 'positive_ratings', 'negative_ratings', 'price_usd', 'discount', 'average_playtime', 'label']

Dtypes:
steamid                    object
appid                       int64
name                       object
playtime_hours            float64
username                   object
country                    object
timecreated                 int64
achievements_completed    float64
achievements_total        float64
steamspy_name              object
genre                      object
tags                       object
owners                     object
positive_ratings            int64
negative_ratings            int64
price_usd                 float64
discount       

/tmp/ipykernel_5167/57016404.py:134: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_frames, ignore_index=True)


In [6]:

# WRANGLING v2: TYPE CONVERSION + DATE PARSING
master_df = pd.read_csv(f"{output_dir}steam_v1_raw.csv")
print(f"Loaded raw data: {master_df.shape}")


# TYPE CONVERSION
print("\n--- Type Conversion ---")

master_df["playtime_hours"] = master_df["playtime_hours"].astype(float)
master_df["achievements_completed"] = master_df["achievements_completed"].fillna(0).astype(int)
master_df["achievements_total"] = master_df["achievements_total"].fillna(0).astype(int)
master_df["positive_ratings"] = pd.to_numeric(master_df["positive_ratings"], errors="coerce").fillna(0).astype(int)
master_df["negative_ratings"] = pd.to_numeric(master_df["negative_ratings"], errors="coerce").fillna(0).astype(int)
master_df["price_usd"] = pd.to_numeric(master_df["price_usd"], errors="coerce").fillna(0).astype(float)
master_df["discount"] = pd.to_numeric(master_df["discount"], errors="coerce").fillna(0).astype(int)

if "average_playtime" in master_df.columns:
    master_df["average_playtime_hours"] = pd.to_numeric(master_df["average_playtime"], errors="coerce").fillna(0) / 60
    master_df = master_df.drop(columns=["average_playtime"])

print(f"Dtypes after conversion:\n{master_df.dtypes}")


# DATE PARSING
print("\n--- Date Parsing ---")

master_df["account_created"] = pd.to_datetime(master_df["timecreated"], unit="s", errors="coerce")
master_df["account_age_years"] = (pd.Timestamp.now() - master_df["account_created"]).dt.days / 365.25
master_df = master_df.drop(columns=["timecreated"])

print(master_df[["steamid", "username", "account_created", "account_age_years"]].drop_duplicates().to_string(index=False))

print(f"\nShape: {master_df.shape}")
master_df.to_csv(f"{output_dir}steam_v2_typed.csv", index=False)
print("Exported: steam_v2_typed.csv")

Loaded raw data: (130, 19)

--- Type Conversion ---
Dtypes after conversion:
steamid                     int64
appid                       int64
name                       object
playtime_hours            float64
username                   object
country                    object
timecreated                 int64
achievements_completed      int64
achievements_total          int64
steamspy_name              object
genre                      object
tags                       object
owners                     object
positive_ratings            int64
negative_ratings            int64
price_usd                 float64
discount                    int64
label                      object
average_playtime_hours    float64
dtype: object

--- Date Parsing ---
          steamid          username     account_created  account_age_years
76561199121994793 codykihlberg21985 2020-12-25 16:13:39           5.311431
76561198244049512             oMatt 2015-08-10 22:42:42          10.685832

Shape: (130, 20

In [7]:

# WRANGLING v3: REGEX + TEXT CLEANING

import re
master_df = pd.read_csv(f"{output_dir}steam_v2_typed.csv")
print(f"Loaded typed data: {master_df.shape}")


# REGEX — Extract lower bound from owners range string

print("\n--- Regex ---")
master_df["owners_lower"] = (
    master_df["owners"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+)")
    .astype(float)
)
print("Sample owners parsing:")
print(master_df[["owners", "owners_lower"]].drop_duplicates().head(5).to_string(index=False))

master_df = master_df.drop(columns=["owners"])

# TEXT CLEANING — Clean game names, flatten tags, fill nulls

print("\n--- Text Cleaning ---")
master_df["name"] = (
    master_df["name"]
    .astype(str)
    .str.strip()
    .str.replace(r"[™®©]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

def flatten_tags(tags):
    if isinstance(tags, str) and tags.startswith("{"):
        try:
            tags_dict = eval(tags)
            if isinstance(tags_dict, dict):
                return ", ".join(list(tags_dict.keys())[:3])
        except:
            pass
    if isinstance(tags, dict):
        return ", ".join(list(tags.keys())[:3])
    return "Unknown"

master_df["top_tags"] = master_df["tags"].apply(flatten_tags)
master_df = master_df.drop(columns=["tags"])

master_df["country"] = master_df["country"].fillna("Unknown")
master_df["genre"] = master_df["genre"].fillna("Unknown")

print(f"Sample cleaned names: {master_df['name'].head(5).tolist()}")
print(f"Sample tags: {master_df['top_tags'].head(5).tolist()}")

print(f"\nShape: {master_df.shape}")
master_df.to_csv(f"{output_dir}steam_v3_cleaned.csv", index=False)
print("Exported: steam_v3_cleaned.csv")

Loaded typed data: (130, 20)

--- Regex ---
Sample owners parsing:
                  owners  owners_lower
20,000,000 .. 50,000,000    20000000.0
 5,000,000 .. 10,000,000     5000000.0
10,000,000 .. 20,000,000    10000000.0
  2,000,000 .. 5,000,000     2000000.0
      100,000 .. 200,000      100000.0

--- Text Cleaning ---
Sample cleaned names: ['Rust', 'Hearts of Iron IV', 'Valheim', 'Total War: WARHAMMER III', 'Sea of Thieves']
Sample tags: ['Survival, Crafting, Multiplayer', 'Strategy, World War II, Grand Strategy', 'Open World Survival Craft, Survival, Online Co-Op', 'Strategy, Turn-Based Strategy, Grand Strategy', 'Multiplayer, Open World, Adventure']

Shape: (130, 20)
Exported: steam_v3_cleaned.csv


In [10]:
# WRANGLING v4: FUZZY MATCHING + DEDUPLICATION

!pip install thefuzz python-Levenshtein -q
from thefuzz import fuzz

master_df = pd.read_csv(f"{output_dir}steam_v3_cleaned.csv")
print(f"Loaded cleaned data: {master_df.shape}")


# FUZZY MATCHING — Compare Steam API name vs SteamSpy name

print("\n--- Fuzzy Matching ---")
master_df["name_match_score"] = master_df.apply(
    lambda row: fuzz.ratio(str(row["name"]), str(row["steamspy_name"]))
    if pd.notna(row["name"]) and pd.notna(row["steamspy_name"])
    else None,
    axis=1
)

mismatches = master_df[master_df["name_match_score"] < 90][
    ["appid", "name", "steamspy_name", "name_match_score"]
].drop_duplicates()

if len(mismatches) > 0:
    print(f"Found {len(mismatches)} name mismatches (score < 90):")
    print(mismatches.head(10).to_string(index=False))
else:
    print("All game names match between Steam API and SteamSpy (score >= 90)")

master_df = master_df.rename(columns={"name": "game_name"})
master_df = master_df.drop(columns=["steamspy_name"])

# DEDUPLICATION — Remove duplicate user+game rows

print("\n--- Deduplication ---")
before = len(master_df)
master_df = master_df.drop_duplicates(subset=["steamid", "appid"])
after = len(master_df)
print(f"Rows before: {before} | After: {after} | Removed: {before - after}")

print(f"\nShape: {master_df.shape}")
master_df.to_csv(f"{output_dir}steam_v4_deduped.csv", index=False)
print("Exported: steam_v4_deduped.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 40.0 MB/s eta 0:00:00
Loaded cleaned data: (130, 20)

--- Fuzzy Matching ---
Found 9 name mismatches (score < 90):
  appid                                     name                            steamspy_name  name_match_score
1172620                           Sea of Thieves             Sea of Thieves: 2026 Edition                67
1938090                             Call of Duty          Call of Duty: Modern Warfare II                56
    730                         Counter-Strike 2         Counter-Strike: Global Offensive                62
   3590     Plants vs. Zombies: Game of the Year          Plants vs. Zombies GOTY Edition                66
   6060 Star Wars: Battlefront 2 (Classic, 2005) STAR WARS Battlefront II (Classic, 2005)                80
 495420                         State of Decay 2     State of Decay 2: Juggernaut Edition             

In [14]:

# WRANGLING v5: RESHAPING + TIDYING + FINAL EXPORT


master_df = pd.read_csv(f"{output_dir}steam_v4_deduped.csv")
print(f"Loaded deduped data: {master_df.shape}")

# RESHAPING — Explode comma-separated genres into long format

print("\n--- Reshaping ---")
master_df["genre_list"] = master_df["genre"].str.split(", ")
genres_long = master_df.explode("genre_list")
genres_long = genres_long.rename(columns={"genre_list": "single_genre"})

print(f"Master df shape (wide): {master_df.shape}")
print(f"Genres long shape (exploded): {genres_long.shape}")
print(f"Unique genres found: {genres_long['single_genre'].nunique()}")
print(f"\nGenre counts:")
print(genres_long["single_genre"].value_counts().head(10).to_string())


# TIDYING — Create a normalized genre lookup table

print("\n--- Tidying ---")
genre_lookup = (
    genres_long[["appid", "game_name", "single_genre"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Genre lookup table: {genre_lookup.shape[0]} rows (one row = one game-genre pair)")
print(genre_lookup.head(10).to_string(index=False))

master_df = master_df.drop(columns=["genre_list"])


# FINAL CLEANUP & EXPORT

master_df = master_df.reset_index(drop=True)

print(f"\n{'='*60}")
print(f"FINAL MASTER DATAFRAME")
print(f"{'='*60}")
print(f"Shape: {master_df.shape}")
print(f"Columns: {list(master_df.columns)}")
print(f"\nMissing values:\n{master_df.isnull().sum()}")
print(f"\nDtypes:\n{master_df.dtypes}")
print(f"\nFirst 5 rows:")
print(master_df.head().to_string())

master_df.to_csv(f"{output_dir}steam_v5_final.csv", index=False)
genre_lookup.to_csv(f"{output_dir}steam_genre_lookup.csv", index=False)
genres_long.to_csv(f"{output_dir}steam_genres_long.csv", index=False)

print("\nExported:")
print(f"  {output_dir}steam_v5_final.csv")
print(f"  {output_dir}steam_genre_lookup.csv")
print(f"  {output_dir}steam_genres_long.csv")

Loaded deduped data: (130, 20)

--- Reshaping ---
Master df shape (wide): (130, 21)
Genres long shape (exploded): (333, 21)
Unique genres found: 18

Genre counts:
single_genre
Action                   81
Adventure                46
Indie                    46
Free To Play             32
Casual                   24
Strategy                 20
Simulation               19
RPG                      17
Sports                   16
Massively Multiplayer    11

--- Tidying ---
Genre lookup table: 326 rows (one row = one game-genre pair)
 appid         game_name          single_genre
252490              Rust                Action
252490              Rust             Adventure
252490              Rust                 Indie
252490              Rust Massively Multiplayer
252490              Rust                   RPG
394360 Hearts of Iron IV            Simulation
394360 Hearts of Iron IV              Strategy
892970           Valheim                Action
892970           Valheim             Advent